> ## ⚠️ SUPERSEDED — do not use
>
> This notebook is **not** the generator of the published figures and its labels
> are out of date: Figure 2 below still says "MeSH genetics filter" and
> "~3.4M articles", whereas the corpus gate is lexical (PMC OA JATS carries no
> MeSH indexing) and the indexed set is 2.25M articles. Figure 1 still shows a
> criterion (iv) box that the manuscript no longer describes.
>
> Use **`scripts/manuscript/render_p1_figures.py`** instead. That script was
> verified against the deployed figures: re-running it reproduces all four PNGs
> byte-for-byte.
>
> Kept only as a record of the earlier exploratory workflow.

# GenoAgent P1 — Figure reproduction notebook

Reproduces every figure of the P1 Data Note from the released cohort and the
pinned HPO inputs:

- **Figure 1** — CONSORT-style cohort selection flow (with the full intake funnel)
- **Figure 2** — deterministic PMC OA hybrid-index build pipeline (schematic)
- **Figure 3** — cohort characterisation (category, overlap, recency, HPO depth)
- **Figure 5** — candidate-list difficulty: *standard* (random) vs *hard*
  (phenotype-similar) distractors, on a common HPO Resnik best-match-average scale

Run top-to-bottom; all counts are computed live from the data. Figures are also
written to `reports/_local/updated-paper-2/` for the manuscript.

In [ ]:
import json, re, importlib.util
from collections import Counter
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# --- repository paths --------------------------------------------------------
ROOT = Path.cwd().resolve()
while not (ROOT / "scripts" / "cases").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
STAGE = ROOT / "figshare_uploads/_staging/genoagent-cohort-n1047-v1.0"
HARD  = ROOT / "figshare_uploads/_staging/genoagent-cohort-hard-n1047-v1.0"
OUT   = ROOT / "reports/figures"                      # canonical, tracked
ALSO  = ROOT / "reports/_local/updated-paper-2"       # Overleaf working copy (if present)
OUT.mkdir(parents=True, exist_ok=True)
def savefig(fig, name):
    fig.savefig(OUT / name, dpi=300, bbox_inches="tight")
    if ALSO.exists():
        fig.savefig(ALSO / name, dpi=300, bbox_inches="tight")
print("repo root:", ROOT)

recs  = [json.loads(l) for l in open(STAGE / "test_cases.jsonl")]
ov    = {r["case_id"]: r["overlap"]
         for r in json.load(open(STAGE / "annotation_overlap.json"))["records"]}
dates = json.load(open(STAGE / "pmid_dates.json"))["dates"]
pmid_of = lambda cid: re.search(r"PMID_(\d+)", cid).group(1)
CATS = ["developmental", "immunological", "metabolic", "neurological"]

plt.rcParams.update({"font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold",
                     "axes.spines.top": False, "axes.spines.right": False,
                     "font.family": "DejaVu Sans"})
print("cases:", len(recs))

## Headline numbers + intake funnel (computed from the released files)

In [ ]:
cat = Counter(r["category"] for r in recs)
n_overlap = sum(ov.values())
years = np.array([int(dates[pmid_of(r["case_id"])][:4]) for r in recs])
hpo_n = np.array([len(r["hpo_terms"]) for r in recs])

# intake funnel from the staged provenance files
wc = lambda p: sum(1 for _ in open(p))
loaded   = wc(STAGE / "01_all_phenopackets.jsonl")   # all phenopackets
elig_ii  = wc(STAGE / "02_eligible.jsonl")           # after criteria (i)-(ii)
categ    = wc(STAGE / "03_categorized.jsonl")        # after criterion (iii)
sampled  = wc(STAGE / "04_sampled.jsonl")            # stratified sample
final    = len(recs)
cat_pool = Counter(json.loads(l)["category"] for l in open(STAGE / "03_categorized.jsonl"))

print("funnel:", loaded, "->", elig_ii, "->", categ, "->", sampled, "->", final)
print("excluded (i-ii):", loaded-elig_ii, "| other MONDO:", elig_ii-categ,
      "| not drawn:", categ-sampled, "| RNU removed:", sampled-final)
print("eligible pool by category:", dict(cat_pool))
print("categories:", {c: cat[c] for c in CATS})
print(f"overlap present/absent: {n_overlap} ({100*n_overlap/len(recs):.1f}%) / "
      f"{len(recs)-n_overlap} ({100*(len(recs)-n_overlap)/len(recs):.1f}%)")
print(f"recency pre/post-2020: {(years<2020).sum()} / {(years>=2020).sum()}")
print(f"pub-year span/median: {years.min()}-{years.max()} / "
      f"{int(np.median([int(v[:4]) for v in dates.values()]))} ({len(dates)} PMIDs)")
print(f"HPO terms median/range: {int(np.median(hpo_n))} / {hpo_n.min()}-{hpo_n.max()}")

## Figure 1 — CONSORT-style cohort selection flow

Now itemises the full intake funnel, including the 9,588 loaded and 6,382
post-inclusion counts with per-step exclusions.

In [ ]:
fig, ax = plt.subplots(figsize=(8.6, 12))
ax.set_xlim(0, 10); ax.set_ylim(0, 15); ax.axis("off")
BLUE  = dict(boxstyle="round,pad=0.5", facecolor="#eaf1f9", edgecolor="#2b6cb0", linewidth=1.5)
EXCL  = dict(boxstyle="round,pad=0.45", facecolor="#fef5e7", edgecolor="#dd6b20", linewidth=1.3)
GREEN = dict(boxstyle="round,pad=0.6", facecolor="#cdeccf", edgecolor="#2f855a", linewidth=1.8)
TAN   = dict(boxstyle="round,pad=0.6", facecolor="#fdf3e3", edgecolor="#c05621", linewidth=1.8)
MX = 3.7  # main column x

def tb(x, y, t, style, fs=9.0, w="normal"):
    ax.text(x, y, t, ha="center", va="center", fontsize=fs, bbox=style, weight=w, zorder=3)
def down(y1, y2, x=MX):
    ax.annotate("", xy=(x, y2), xytext=(x, y1), arrowprops=dict(arrowstyle="-|>", color="#2b6cb0", lw=1.6))
def side(y, n):
    # arrow start nudged to 6.2 (was 6.0) so it clears the widest main box
    ax.annotate("", xy=(6.9, y), xytext=(6.2, y), arrowprops=dict(arrowstyle="-|>", color="#dd6b20", lw=1.2))
    tb(8.3, y, n, EXCL, fs=8.0)

tb(MX, 14.2, "GA4GH Phenopacket Store v0.1.26\n(public; literature-curated, gene-level SOLVED)\n"
   f"N = {loaded:,} phenopackets loaded", BLUE)
down(13.55, 12.95)
tb(MX, 12.3, "Inclusion criteria (i)-(ii):\nsingle SOLVED causal gene; >=3 HPO terms\n"
   f"N = {elig_ii:,}", BLUE)
side(12.3, f"Excluded:\nnon-SOLVED or\n<3 HPO terms\n(N = {loaded-elig_ii:,})")
down(11.7, 11.0)
tb(MX, 10.3, "Categorise into 4 MONDO supercategories (iii)\n"
   f"Eligible pool  N = {categ:,}\n"
   f"({cat_pool['developmental']} dev / {cat_pool['immunological']} imm / "
   f"{cat_pool['metabolic']} met / {cat_pool['neurological']:,} neu)", BLUE)
side(10.3, f"Excluded:\nother MONDO\ncategories\n(N = {elig_ii-categ:,})")
down(9.55, 8.75)
tb(MX, 8.0, "Disproportionate stratified sample (seed 42)\ntarget 250 / 300 / 250 / 250\n"
   "criterion (iv): >=5 PMC OA articles\n"
   f"Drawn  N = {sampled:,} (verified; no reduction)", BLUE)
side(8.0, f"Not drawn by\nstratified sampling\n(N = {categ-sampled:,})")
down(7.15, 6.35)
tb(MX, 5.7, "Remove 3 non-protein-coding RNA causal genes\n(2x RNU4-2, 1x RNU2-2) at candidate-list stage", BLUE)
down(5.2, 4.55)
tb(MX, 3.9, f"ANALYTIC COHORT   n = {final:,}\n(250 dev / 300 imm / 250 met / 247 neu)", GREEN, fs=9.5, w="bold")
down(3.25, 2.6)
tb(MX, 1.85, "Annotation-overlap stratification (per-PMID vs phenotype.hpoa)\n"
   f"Overlap-present  n = {n_overlap} ({100*n_overlap/final:.1f}%)\n"
   f"Overlap-absent (FAIR COHORT)  n = {final-n_overlap} ({100*(final-n_overlap)/final:.1f}%)", TAN)
ax.set_title("Figure A1 - CONSORT-style cohort selection flow", fontsize=12, fontweight="bold", pad=8)
savefig(fig, "fig1_consort_flow.png")
plt.show()

## Figure 2 — Deterministic PMC OA hybrid-index build pipeline

A raster reproduction of the manuscript's TikZ schematic (Fig 2).

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.set_xlim(0, 12); ax.set_ylim(0, 4); ax.axis("off")
PB = dict(boxstyle="round,pad=0.45", facecolor="#eaf1f9", edgecolor="#2b6cb0", linewidth=1.5)
PG = dict(boxstyle="round,pad=0.45", facecolor="#cdeccf", edgecolor="#2f855a", linewidth=1.6)
def pb(x, y, t, style, fs=8.6):
    ax.text(x, y, t, ha="center", va="center", fontsize=fs, bbox=style, zorder=3)
def ar(x1, y1, x2, y2, c="#2b6cb0"):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="-|>", color=c, lw=1.6))

pb(1.5, 2.6, "PMC OA full text\nMeSH genetics filter\n(~3.4M articles)", PB)
pb(4.2, 2.6, "Section-aware chunking\n512 tok / 50 overlap", PB)
pb(7.0, 2.6, "PubMedBERT dense (768-d)\n+ BM25 sparse", PB)
pb(9.9, 2.6, "Qdrant index\n52,777,395 chunks\nHNSW . cosine", PG)
pb(7.0, 0.8, "UUID5 content IDs\n(deterministic)", PB)
pb(9.9, 0.8, "Hybrid retrieval\n(RRF, k=60)", PG)
ar(2.55, 2.6, 3.15, 2.6); ar(5.3, 2.6, 5.85, 2.6); ar(8.1, 2.6, 8.9, 2.6)
ar(7.0, 1.25, 9.3, 2.2)          # uuid -> qdrant
ar(9.9, 2.2, 9.9, 1.25)          # qdrant -> retrieval
ax.text(6.0, 3.7, "deterministic, version-pinned, bit-for-bit reproducible",
        ha="center", va="center", fontsize=8.5, style="italic", color="#555")
ax.set_title("Figure 2 - Deterministic PMC OA hybrid-index build pipeline",
             fontsize=12, fontweight="bold", pad=6)
savefig(fig, "fig2_index_pipeline.png")
plt.show()

## Figure 3 — Cohort characterisation

In [ ]:
C = {"developmental": "#4C72B0", "immunological": "#DD8452",
     "metabolic": "#55A868", "neurological": "#C44E52"}
PRESENT, ABSENT = "#B0B0B0", "#2C7FB8"
fig, ax = plt.subplots(2, 2, figsize=(10.5, 8.4))

# (a) category balance
vals = [cat[c] for c in CATS]
bars = ax[0,0].bar(range(4), vals, color=[C[c] for c in CATS], width=0.68)
ax[0,0].set_xticks(range(4)); ax[0,0].set_xticklabels([c.capitalize() for c in CATS], rotation=20, ha="right")
ax[0,0].set_ylabel("Cases"); ax[0,0].set_title("(a) Disease-category balance"); ax[0,0].set_ylim(0, 345)
for b, v in zip(bars, vals): ax[0,0].text(b.get_x()+b.get_width()/2, v+6, str(v), ha="center", fontsize=9)

# (b) overlap split -- counts folded into legend labels, headroom above bars (no overlap)
absent  = [sum(1 for r in recs if r["category"]==c and ov[r["case_id"]]==0) for c in CATS]
present = [sum(1 for r in recs if r["category"]==c and ov[r["case_id"]]==1) for c in CATS]
tot_p, tot_a = sum(present), sum(absent)
ax[0,1].bar(range(4), present, color=PRESENT, width=0.68,
            label=f"Overlap-present - leakage risk  ({tot_p}, {100*tot_p/final:.1f}%)")
ax[0,1].bar(range(4), absent, bottom=present, color=ABSENT, width=0.68,
            label=f"Overlap-absent - fair subset  ({tot_a}, {100*tot_a/final:.1f}%)")
ax[0,1].set_xticks(range(4)); ax[0,1].set_xticklabels([c[:5].capitalize() for c in CATS], rotation=20, ha="right")
ax[0,1].set_ylabel("Cases"); ax[0,1].set_title("(b) Annotation-overlap split"); ax[0,1].set_ylim(0, 380)
ax[0,1].legend(fontsize=7.6, loc="upper center", frameon=False)

# (c) recency -- annotation moved to empty upper-left, away from the bars
ax[1,0].hist(years, bins=np.arange(years.min(), years.max()+2), color="#7A7A7A", edgecolor="white", linewidth=0.4)
ax[1,0].axvline(2020, color="#C44E52", ls="--", lw=1.4)
ax[1,0].text(0.02, 0.95, f"2020 split:  pre {(years<2020).sum()} / post {(years>=2020).sum()}",
             transform=ax[1,0].transAxes, ha="left", va="top", color="#C44E52", fontsize=8.5)
ax[1,0].set_xlabel("Source-publication year"); ax[1,0].set_ylabel("Cases")
ax[1,0].set_title("(c) Publication-recency distribution"); ax[1,0].xaxis.set_major_locator(MaxNLocator(integer=True, nbins=8))

# (d) HPO depth
ax[1,1].hist(hpo_n, bins=np.arange(hpo_n.min(), hpo_n.max()+2), color="#4C72B0", edgecolor="white", linewidth=0.4)
med = int(np.median(hpo_n)); ax[1,1].axvline(med, color="#DD8452", ls="--", lw=1.4)
ax[1,1].text(0.97, 0.95, f"median {med}\n(range {hpo_n.min()}-{hpo_n.max()})",
             transform=ax[1,1].transAxes, ha="right", va="top", color="#DD8452", fontsize=8.5)
ax[1,1].set_xlabel("HPO terms per case"); ax[1,1].set_ylabel("Cases"); ax[1,1].set_title("(d) Phenotype-annotation depth")
fig.tight_layout(pad=1.5)
savefig(fig, "fig3_cohort_characterisation.png")
plt.show()

## Figure 5 — Candidate-list difficulty (standard vs hard)

Uses the *exact* Resnik/BMA machinery from `scripts/cases/18b_build_hard_candidates.py`.

In [ ]:
spec = importlib.util.spec_from_file_location("hb", ROOT / "scripts/cases/18b_build_hard_candidates.py")
hb = importlib.util.module_from_spec(spec); spec.loader.exec_module(hb)
parents, alt = hb.parse_hpo_obo(hb.HPO_DIR / "hp.obo"); valid = set(parents)
anc = hb.build_ancestors(parents)
gene_hpo, _ = hb.load_gene_annotations(hb.HPO_DIR / "genes_to_phenotype.txt", alt, valid)
ic = hb.compute_ic(gene_hpo, anc); resnik = hb.make_similarity(anc, ic)
case_terms = lambda r: [t for t in (alt.get(x, x) for x in r["hpo_terms"]) if t in valid]

std  = {r["case_id"]: r for r in recs}
hard = {json.loads(l)["case_id"]: json.loads(l) for l in open(HARD / "test_cases_hard.jsonl")}
causal_bma, rand_all, hard_all, rand_hardest, hard_hardest = [], [], [], [], []
for cid, r in std.items():
    ct = case_terms(r); causal = r["causal_gene"]
    causal_bma.append(hb.bma(ct, gene_hpo.get(causal, frozenset()), resnik))
    rb  = [hb.bma(ct, gene_hpo.get(g, frozenset()), resnik) for g in r["candidate_genes"] if g != causal]
    hbm = [hb.bma(ct, gene_hpo.get(g, frozenset()), resnik) for g in hard[cid]["candidate_genes"] if g != causal]
    rand_all += rb; hard_all += hbm; rand_hardest.append(max(rb)); hard_hardest.append(max(hbm))
causal_bma, rand_all, hard_all = map(np.array, (causal_bma, rand_all, hard_all))
rand_hardest, hard_hardest = np.array(rand_hardest), np.array(hard_hardest)
rand_frac = float((rand_hardest >= causal_bma).mean()); hard_frac = float((hard_hardest >= causal_bma).mean())
print(f"median BMA causal {np.median(causal_bma):.2f} | random {np.median(rand_all):.2f} | hard {np.median(hard_all):.2f}")
print(f"ties/exceeds causal: random {100*rand_frac:.1f}% vs hard {100*hard_frac:.1f}%")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4.3))
hi = max(hard_all.max(), causal_bma.max()); b = np.linspace(0, hi, 40)
ax[0].hist(rand_all, bins=b, density=True, color="#9E9E9E", alpha=0.8, label="Random distractors (DOI .449)")
ax[0].hist(hard_all, bins=b, density=True, color="#DD8452", alpha=0.65, label="Hard distractors (phenotype-similar)")
ax[0].axvline(np.median(causal_bma), color="#C44E52", ls="--", lw=1.6, label=f"Causal gene (median {np.median(causal_bma):.2f})")
ax[0].set_xlabel("Case-gene phenotypic similarity (Resnik BMA)"); ax[0].set_ylabel("Density")
ax[0].set_title("(a) Distractor phenotypic similarity")
ax[0].legend(fontsize=7.6, frameon=False, loc="upper right")   # empty upper-right region
ax[1].scatter(rand_hardest, causal_bma, s=7, alpha=0.30, color="#9E9E9E", edgecolors="none",
              label=f"Random (ties causal: {100*rand_frac:.0f}%)")
ax[1].scatter(hard_hardest, causal_bma, s=7, alpha=0.30, color="#DD8452", edgecolors="none",
              label=f"Hard (ties causal: {100*hard_frac:.0f}%)")
lim = max(hard_hardest.max(), causal_bma.max())*1.05
ax[1].plot([0, lim], [0, lim], color="0.35", ls="--", lw=1); ax[1].set_xlim(0, lim); ax[1].set_ylim(0, lim)
ax[1].set_xlabel("Hardest distractor similarity (max BMA)"); ax[1].set_ylabel("Causal-gene similarity (BMA)")
ax[1].set_title("(b) Per-case separability"); ax[1].legend(fontsize=8, frameon=False, loc="lower right")
fig.tight_layout(pad=1.2)
savefig(fig, "fig4_hard_vs_random_separability.png")
plt.show()

---
*All figures regenerate deterministically from the released cohort, the pinned HPO
inputs (v2026-02-16), and `scripts/cases/18b_build_hard_candidates.py`. Figures are
written to `reports/_local/updated-paper-2/`.*